This is a standalone graphing examples to compare with the results found in https://arxiv.org/pdf/2604.06304

In [1]:
# get data from all_particle_data
import numpy as np
import matplotlib.pyplot as plt
n_snapshots = 100+1
npts = 650000

dt = np.dtype([('rank','i4'),('R','f4'),('Vrad','f4'),('L','f4'),('species','i4')])
data = np.fromfile("/home/posner/NSphere/data/all_particle_data_650000_10001_0.1.dat",dtype = dt).reshape(n_snapshots, npts)

rad = data["R"]
species = data["species"]
Vrad = data["Vrad"]

print(data.shape)

(101, 650000)


In [2]:
# get time array

def get_timestamps(tag = "", suffix = "100000_10001_5"):
    "Reads the single_trajectory files from the data directory and constructs a data array from them"

    #create path path
    if tag == "":
        file_path = "../data/single_trajectory" + "_" + suffix + ".dat"
    else:
        file_path = "../data/single_trajectory" + "_" + tag + "_" + suffix + ".dat"

    #define data types
    record_dtype = np.dtype([
        ('time',  np.float32),
        ('R',     np.float32),
        ('Vrad',  np.float32),
        ('mu',  np.float32),
    ])
    
    #read the files and build data array
    try:
        # Read the binary file and extract radius data
        alldata = np.fromfile(file_path, dtype=record_dtype)
        data = alldata['R']

        if np.isnan(data).any():
            print(f"Warning: NaN values found in data at Time Step: {time_step}")
        if not np.isfinite(data).all():
            print(f"Warning: Infinite values found in data at Time Step: {time_step}")

    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
    return alldata["time"]

physical_time = get_timestamps(suffix = "650000_10001_0.1")
time_total = physical_time[-1] - physical_time[0]
delta_t = time_total / n_snapshots

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --------------------------------------------------
# Simulation parameters
# --------------------------------------------------

# Replace with your NFW parameters
Mh = 792.0      # Msun
rs = 1.18       # kpc
c = 19.0        # Concentration (falloff) factor

f = np.log(1 + c) - c/(1 + c)
rho_s = Mh / (4*np.pi*rs**3*f)

# Radius range for density profiles
rmin = np.nanmin(rad[rad > 0])
rmax = np.nanmax(rad)

nbins = 60
bin_edges = np.logspace(np.log10(rmin), np.log10(rmax), nbins + 1)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])

# --------------------------------------------------
# NFW profile
# --------------------------------------------------

def rho_nfw(r):
    x = r / rs
    return rho_s / (x * (1 + x)**2)

rho_theory = rho_nfw(bin_centers)

# --------------------------------------------------
# Density estimator
# --------------------------------------------------

def density_profile(radii):
    """
    Compute density in spherical shells.
    Assumes every particle has equal mass.
    """
    counts, _ = np.histogram(radii, bins=bin_edges)

    shell_volumes = (4.0/3.0) * np.pi * (
        bin_edges[1:]**3 - bin_edges[:-1]**3
    )

    density = counts / shell_volumes
    density[density == 0] = np.nan

    return density

# --------------------------------------------------
# Figure
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(8,6))

colors = {
    0: "tab:blue",
    1: "tab:orange",
}

labels = {
    0: "Species 0",
    1: "Species 1",
}

species_values = np.unique(species)

# --------------------------------------------------
# Animation function
# --------------------------------------------------

def update(frame):

    ax.clear()

    # Plot theoretical profile
    ax.plot(
        bin_centers,
        rho_theory,
        color="black",
        lw=2,
        label="NFW"
    )

    # Plot each species
    for sp in species_values:

        mask = species[frame] == sp

        rho = density_profile(rad[frame][mask])

        ax.plot(
            bin_centers,
            rho,
            lw=2,
            color=colors.get(sp, None),
            label=labels.get(sp, f"Species {sp}")
        )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Radius")
    ax.set_ylabel("Density")

    ax.set_title(f"t = {delta_t*frame/1000:.3f} Gyr")

    ax.set_xlim(1e-3, 1e2) 
    ax.set_ylim(1e-5, 1e7)

    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

# --------------------------------------------------
# Animation
# --------------------------------------------------

ani = FuncAnimation(
    fig,
    update,
    frames=n_snapshots,
    interval=100,
    blit=False
)
plt.close(fig)

# Uncomment this line to save the animation as a GIF
#ani.save("../results/nsphere_density_Energy_lost.gif", writer="pillow", fps=10)

HTML(ani.to_jshtml())